# Decorators
Decorators are functions which take a function as input and return a function.
You have probably already since them with the `@decorator` syntax, and in particular `@property`

## Decorators 101

Replace a function... by anything else...

In [ ]:
def my_pointless_decorator(func):
    print(f"I'm modifying func: {func}")
    return 12

In [ ]:
@my_pointless_decorator
def a_func():
    print("Hello")

In [ ]:
print(f"This is now a_func: {a_func}")

In [ ]:
# IMPORTANT: it's just syntactic sugar for this:

def a_func():
    print("Hello")

print(f"This is a_func before decorating: {a_func}")

a_func = my_pointless_decorator(a_func)
print(f"The output of the decorator is: {a_func}")

In [ ]:
import typing as tp
import time

def my_decorator(func: tp.Callable[..., tp.Any]) -> tp.Callable[..., tp.Any]:
    print("Decorating - start")
    
    def new_func(*args, **kwargs):
        t0 = time.time()
        print(f"Calling with {args}, {kwargs}")
        out = func(*args, **kwargs)
        print(f"Calling - end after {time.time()-t0:1.6f}s")
        return out
    
    print("Decorating - end")
    return new_func  # we return the new function instead of the initial one! We replace it!

In [ ]:
def my_func(value: int) -> int:
    return value + 12

In [ ]:
my_func(2)

In [ ]:
@my_decorator
def my_func(value: int) -> int:
    return value + 12

In [ ]:
my_func(2)

In [ ]:
my_func(2)

## More decorator patterns

In [ ]:
import pickle
pickle.dumps(my_func)  # oh no...

In [ ]:
# a better pattern for writing decorators

class my_new_decorator:  # only reasonable time to not use MyClass naming pattern
    
    def __init__(self, func: tp.Callable[..., tp.Any]) -> None:
        self._func = func
    
    def __call__(self, *args: tp.Any, **kwargs: tp.Any) -> tp.Any:
        print(f"Calling with {args}, {kwargs}")
        out = self._func(*args, **kwargs)
        print("Calling - end")
        return out
        

In [ ]:
@my_new_decorator
def my_func(value: int) -> int:
    return value + 12

my_func(2)

In [ ]:
pickle.dumps(my_func)
# ok, not better, but actually it is
# (as long as you define it in an imported module/package)

In [ ]:
# a decorator with arguments!

class my_parametrized_decorator:
    # only reasonable time to not use MyClass naming pattern
    
    def __init__(self, string: str) -> None:
        self._string = string
        self._func: tp.Callable[..., tp.Any] | None = None
    
    def __call__(self, func: tp.Callable[..., tp.Any]) -> tp.Callable[..., tp.Any]:
        self._func = func
        return self._replacement
    
    def _replacement(self, *args: tp.Any, **kwargs: tp.Any) -> tp.Any:
        assert self._func is not None, "Missing decorated function"
        print("Decorator parameter:", self._string)
        print(f"Calling with {args}, {kwargs}")
        out = self._func(*args, **kwargs)
        print("Calling - end")
        return out
        

In [ ]:
@my_parametrized_decorator("Hello World")
def my_func(value: int) -> int:
    return value + 12

my_func(2)

## Properties

In [ ]:
class MyClass:
    def __init__(self) -> None:
        self.my_attribute = 12
        self._calls = 0
    
    @property
    def num_calls(self) -> int:
        self._calls += 1
        return self._calls

In [ ]:
instance = MyClass()
instance.my_attribute

In [ ]:
instance.num_calls

In [ ]:
instance.num_calls

In [ ]:
instance.my_attribute = 3
instance.my_attribute

In [ ]:
instance.num_calls = 3

In [ ]:
# let's add a setter

class MyClass:
    def __init__(self) -> None:
        self.my_attribute = 12
        self._called = 0
    
    @property
    def num_calls(self) -> int:
        self._calls += 1
        return self._calls
    
    @num_calls.setter  # syntax is weird... but just remember it for now (names must match)
    def num_calls(self, value: int) -> int:
        if value < 0:
            raise ValueError("Negative numbers are not allowed")
        self._calls = value

In [ ]:
instance = MyClass()
instance.num_calls = 12
instance.num_calls

In [ ]:
instance.num_calls = -1

In [ ]:
# other option:

class MyClass:
    def __init__(self) -> None:
        self.my_attribute = 12
        self._called = 0
    
    
    def _calls_getter(self) -> int:
        self._calls += 1
        return self._calls
    
    def _calls_setter(self, value: int) -> int:
        if value < 0:
            raise ValueError("Negative numbers are not allowed")
        self._calls = value
        
    num_calls = property(_calls_getter, _calls_setter)
    # num_calls is actually a special type of object (a descriptor...  but that's for waaaay later)
    # actually there's a third argument (deleter) for calling `del instance.num_calls`

In [ ]:
instance = MyClass()
instance.num_calls = 3
instance.num_calls

## Static and class methods

In [ ]:
class MyClass:
    
    def method(*args, **kwargs) -> None:
        print("method:", args, kwargs)
    
    @staticmethod
    def staticm(*args, **kwargs) -> None:
        print("static method:", args, kwargs)
    
    @classmethod
    def classm(*args, **kwargs) -> None:
        print("class method:", args, kwargs)
        
inst = MyClass()
inst.method(blublu=12)
inst.classm(blublu=12)
inst.staticm(blublu=12)
    

## Going beyondGoing beyond

 - using `functools.wraps` to keep track of the signature of the initial function
 - properties work using the descriptor pattern... this is the same pattern that makes `self` magically appear in methods :D